# How many corrupted sites does AHA-Fed need to matter?

Every experiment so far corrupts exactly one of four clients. Real
federations may have several unreliable sites, and volume-weighted
averaging should degrade faster as that number grows. This sweeps
1, 2 and 3 corrupted clients out of four, ten seeds each.


In [ ]:
import glob, os, shutil, sys
REPO='/kaggle/working/repo'
hits=glob.glob('/kaggle/input/**/paper/build_paper.py', recursive=True)
assert hits, 'attach the aha-fed-code dataset'
src=os.path.dirname(os.path.dirname(hits[0]))
if not os.path.isdir(os.path.join(REPO,'paper')):
    shutil.copytree(src, REPO, dirs_exist_ok=True)
os.chdir(REPO); sys.path.insert(0, REPO)
print('staged:', os.getcwd())


In [ ]:
!pip install -q medmnist opacus


In [ ]:
import subprocess, sys
PROBE=('import torch;'
       'a=torch.randn(4,3,8,8,device="cuda");'
       'w=torch.randn(2,3,3,3,device="cuda");'
       'torch.nn.functional.conv2d(a,w).sum().item();'
       'print(torch.cuda.get_device_name(0), torch.__version__)')
r=subprocess.run([sys.executable,'-c',PROBE],capture_output=True,text=True)
print(r.stdout.strip() or r.stderr.strip()[-400:])
if r.returncode!=0:
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    'torch==2.4.1','torchvision==0.19.1',
                    '--index-url','https://download.pytorch.org/whl/cu118'],check=True)
    r=subprocess.run([sys.executable,'-c',PROBE],capture_output=True,text=True)
    print(r.stdout.strip() or r.stderr.strip()[-400:])
    if r.returncode!=0: raise SystemExit('no working torch/GPU combination')


In [ ]:
from paper.run_real_experiments import run_one, OUT_DIR

SEEDS=[42,123,456,7,13,21,99,2024,31337,555]
PATH={'dataset':'pathmnist','num_classes':9,'samples_per_client':1000}

for nn in [1,2,3]:
    for s in SEEDS:
        run_one(OUT_DIR/f'multi_fedavg_n{nn}_s{s}.json', aggregator='fedavg',
                num_noisy=nn, seed=s, **PATH)
        run_one(OUT_DIR/f'multi_aha_n{nn}_s{s}.json', aggregator='adaptive',
                alpha=1.0, beta=4.0, gamma=4.0, num_noisy=nn, seed=s, **PATH)
    print(f'{nn} corrupted client(s) done')


In [ ]:
import glob, json, re, statistics as st
from pathlib import Path
from scipy.stats import wilcoxon

def by_seed(pat, key='ece'):
    out={}
    for f in sorted(glob.glob(pat)):
        d=json.loads(Path(f).read_text())
        fin=d.get('final') or {}
        m=re.search(r'_s(\d+)$', Path(f).stem)
        if fin.get(key) is not None:
            out[m.group(1) if m else '42']=fin[key]
    return out

print('corrupted   AHA-Fed      FedAvg       ratio    p')
for nn in [1,2,3]:
    a=by_seed(f'runs/paper/real/multi_aha_n{nn}_s*.json')
    v=by_seed(f'runs/paper/real/multi_fedavg_n{nn}_s*.json')
    c=sorted(set(a)&set(v))
    if not c: continue
    x=[a[k] for k in c]; y=[v[k] for k in c]
    p=wilcoxon(x,y).pvalue
    print(f'  {nn} of 4    {st.mean(x):.4f}      {st.mean(y):.4f}      '
          f'{st.mean(y)/st.mean(x):.2f}x   p={p:.4f}  (n={len(c)})')


In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/multi_results.zip runs/paper/real
!ls -lh /kaggle/working/multi_results.zip
